In [42]:
import polars as pl
from scripts.database import engine

In [43]:

query = """
    SELECT 
        m.ticker, m.date::DATE AS ReportDate, meta.Industry,
        m.Volume, m.Delivery_Percentage, m.daily_hl_spread, m.daily_vwap_dev,
        m.oi_pcr, m.delta_oi_pcr, m.futures_basis, m.net_block_volume, m.avg_block_premium
    FROM mv_unified_market_matrix m
    INNER JOIN market_metadata meta ON m.ticker = meta.Ticker
"""

features = [
    "volume", "delivery_percentage", "daily_hl_spread", "daily_vwap_dev",
    "oi_pcr", "delta_oi_pcr", "futures_basis", "net_block_volume", "avg_block_premium"
]
target_industry = "Aerospace & Defense"
target_ticker = "COCHINSHIP"

# --- Check 2: Industry Beta + Ticker Alpha ---
with engine.stream_lazy(query) as stream_1:
    check_1 = (
        pl.from_arrow(stream_1).lazy()
        .filter(pl.col("Industry") == target_industry)
        .with_columns([
            ((pl.col(f) - pl.col(f).mean().over("ticker")) /
             pl.when(pl.col(f).std().over("ticker") == 0)
             .then(1.0)
             .otherwise(pl.col(f).std().over("ticker"))
            ).alias(f)
            for f in features
        ])
        .select(features)
        .collect(streaming=True).corr()
    )

 #--- Check 3: Ticker Beta + Ticker Alpha ---
with engine.stream_lazy(query) as stream_2:
    check_2 = (
        pl.from_arrow(stream_2).lazy()
        .filter(pl.col("ticker") == target_ticker)
        .select(features)
        .collect(streaming=True).corr()
    )
    

with engine.stream_lazy(query) as stream_3:
  
      
      base_lf = pl.from_arrow(stream_3).lazy().filter(pl.col("Industry") == target_industry)

      industry_factor_lf = (
          base_lf          .group_by("ReportDate")
          .agg([
              pl.col(f).mean().alias(f"{f}_industry")
              for f in features
          ])
      )

      ticker_vs_industry = (
          base_lf
          .filter(pl.col("ticker") == target_ticker)
          .join(industry_factor_lf, on="ReportDate", how="inner")
          .collect(streaming=True)
      )

      check_3 = ticker_vs_industry.select(
          features + [f"{f}_industry" for f in features]
      ).corr()


display("Check 2 (Industry Fixed Effects):\n", check_1)
display("Check 3 (Pure Ticker):\n", check_2)
display(f"Check 3 {target_ticker} vs {target_industry} Baseline:\n", check_3)

C:\Users\Ashish Maharana\AppData\Local\Temp\ipykernel_14204\3608790450.py:31: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True).corr()
C:\Users\Ashish Maharana\AppData\Local\Temp\ipykernel_14204\3608790450.py:40: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True).corr()
C:\Users\Ashish Maharana\AppData\Local\Temp\ipykernel_14204\3608790450.py:61: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


'Check 2 (Industry Fixed Effects):\n'

volume,delivery_percentage,daily_hl_spread,daily_vwap_dev,oi_pcr,delta_oi_pcr,futures_basis,net_block_volume,avg_block_premium
f64,f64,f64,f64,f64,f64,f64,f64,f64
1.0,-0.3057,0.454704,-0.126892,0.104445,0.009381,0.062949,0.010386,-0.167848
-0.3057,1.0,-0.327091,0.06727,0.001467,-0.007477,0.088814,-0.031915,0.052445
0.454704,-0.327091,1.0,0.034772,-0.046722,-0.034333,-0.096183,0.032414,-0.129242
-0.126892,0.06727,0.034772,1.0,-0.055734,-0.1055,-0.024535,-0.058581,0.285496
0.104445,0.001467,-0.046722,-0.055734,1.0,0.136632,0.39953,-0.002224,0.004309
0.009381,-0.007477,-0.034333,-0.1055,0.136632,1.0,0.103469,-0.002039,-0.011761
0.062949,0.088814,-0.096183,-0.024535,0.39953,0.103469,1.0,0.001329,0.007013
0.010386,-0.031915,0.032414,-0.058581,-0.002224,-0.002039,0.001329,1.0,-0.122439
-0.167848,0.052445,-0.129242,0.285496,0.004309,-0.011761,0.007013,-0.122439,1.0


'Check 3 (Pure Ticker):\n'

volume,delivery_percentage,daily_hl_spread,daily_vwap_dev,oi_pcr,delta_oi_pcr,futures_basis,net_block_volume,avg_block_premium
f64,f64,f64,f64,f64,f64,f64,f64,f64
1.0,-0.486032,0.572254,-0.147945,0.048867,-0.005996,0.028225,-0.006289,-0.254237
-0.486032,1.0,-0.327035,0.071326,-0.132766,0.006211,-0.055885,0.017133,0.054509
0.572254,-0.327035,1.0,-0.034845,-0.001757,-0.010768,-0.007709,0.057952,-0.236065
-0.147945,0.071326,-0.034845,1.0,-0.042574,-0.036249,-0.044014,-0.072968,0.367803
0.048867,-0.132766,-0.001757,-0.042574,1.0,0.049229,0.479709,-0.00368,0.006498
-0.005996,0.006211,-0.010768,-0.036249,0.049229,1.0,0.228612,-0.000012,0.000021
0.028225,-0.055885,-0.007709,-0.044014,0.479709,0.228612,1.0,-0.001593,0.002813
-0.006289,0.017133,0.057952,-0.072968,-0.00368,-0.000012,-0.001593,1.0,-0.150861
-0.254237,0.054509,-0.236065,0.367803,0.006498,0.000021,0.002813,-0.150861,1.0


'Check 3 COCHINSHIP vs Aerospace & Defense Baseline:\n'

volume,delivery_percentage,daily_hl_spread,daily_vwap_dev,oi_pcr,delta_oi_pcr,futures_basis,net_block_volume,avg_block_premium,volume_industry,delivery_percentage_industry,daily_hl_spread_industry,daily_vwap_dev_industry,oi_pcr_industry,delta_oi_pcr_industry,futures_basis_industry,net_block_volume_industry,avg_block_premium_industry
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1.0,-0.486032,0.572254,-0.147945,0.048867,-0.005996,0.028225,-0.006289,-0.254237,0.525275,-0.302015,0.265499,-0.12521,0.26492,0.025988,0.141989,0.011357,-0.176834
-0.486032,1.0,-0.327035,0.071326,-0.132766,0.006211,-0.055885,0.017133,0.054509,-0.391486,0.644812,-0.072092,0.120355,-0.308536,-0.032112,-0.257276,-0.017102,0.065032
0.572254,-0.327035,1.0,-0.034845,-0.001757,-0.010768,-0.007709,0.057952,-0.236065,0.385844,-0.20867,0.741866,-0.003893,0.078239,-0.02857,0.00108,0.022826,-0.163253
-0.147945,0.071326,-0.034845,1.0,-0.042574,-0.036249,-0.044014,-0.072968,0.367803,-0.058486,0.049284,0.032838,0.742663,-0.041897,-0.086945,-0.04672,-0.006809,0.278236
0.048867,-0.132766,-0.001757,-0.042574,1.0,0.049229,0.479709,-0.00368,0.006498,0.117864,-0.207594,0.008361,-0.052563,0.517025,0.024279,0.458771,0.008899,-0.009722
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.26492,-0.308536,0.078239,-0.041897,0.517025,0.035237,0.257584,-0.020491,-0.031999,0.347994,-0.280708,-0.030842,-0.107514,1.0,0.175078,0.396465,0.004461,-0.06122
0.025988,-0.032112,-0.02857,-0.086945,0.024279,0.168576,0.03534,-0.004176,-0.000923,0.026696,-0.04635,-0.054092,-0.177199,0.175078,1.0,0.014532,-0.002285,-0.029911
0.141989,-0.257276,0.00108,-0.04672,0.458771,0.188377,0.763552,-0.009394,-0.020164,0.173187,-0.204135,-0.086188,-0.051573,0.396465,0.014532,1.0,0.004613,-0.015836
